# Rolling and Exponentially-Weighted Statistics

screamer ships a broad family of rolling-window and exponentially-weighted
estimators.  All of them share the same callable interface: construct once
with window/span parameters, then call with an array *or* feed scalars
one-by-one.  This notebook surveys the most commonly used members of the
family, shows how the `start_policy` argument controls warm-up behaviour,
and cross-checks screamer's rolling mean against pandas to confirm
numerical equivalence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from screamer import RollingMean, RollingStd, RollingZscore, EwMean, EwStd

rng = np.random.default_rng(0)
price = 100 + np.cumsum(rng.standard_normal(300))

## Rolling vs exponentially-weighted smoothing

`RollingMean(n)` applies an equal-weight window of *n* samples.
`EwMean(span=n)` decays older values geometrically - it is more responsive
to recent changes while still attenuating noise.

In [ ]:
ma20 = RollingMean(20)(price)
ew20 = EwMean(span=20)(price)
z50  = RollingZscore(50)(price)

fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax[0].plot(price, lw=0.6, label="price")
ax[0].plot(ma20, label="RollingMean(20)")
ax[0].plot(ew20, label="EwMean(span=20)")
ax[0].legend(loc="upper left")
ax[1].plot(z50, label="RollingZscore(50)")
ax[1].axhline(0, color="k", lw=0.5)
ax[1].legend()
plt.tight_layout()

## `start_policy`: controlling warm-up behaviour

During the first `n − 1` samples a rolling window is not yet full.
The `start_policy` constructor argument decides what to emit:

| policy | behaviour |
|--------|-----------|
| `"strict"` (default) | emit `NaN` until the window is full |
| `"expanding"` | shrink the denominator - output from sample 1 onward |
| `"zero"` | treat missing history as zeros |

In [ ]:
strict    = RollingMean(20, "strict")(price)
expanding = RollingMean(20, "expanding")(price)
zero      = RollingMean(20, "zero")(price)

# strict emits NaN during warm-up; expanding fills from the very first sample
assert np.isnan(strict[0]) and not np.isnan(expanding[0])
assert np.isnan(strict[18]) and not np.isnan(strict[19])   # full at index 19

print("strict[0:3]    :", strict[:3])
print("expanding[0:3] :", expanding[:3])
print("zero[0:3]      :", zero[:3])

## Cross-check against pandas

`RollingMean` matches pandas `rolling().mean()` for the default `"strict"`
policy (both emit `NaN` during warm-up):

In [ ]:
pd_ma = pd.Series(price).rolling(20).mean().to_numpy()
np.testing.assert_allclose(ma20, pd_ma, equal_nan=True)
print("screamer RollingMean == pandas rolling(20).mean() ✓")

## Rolling standard deviation

`RollingStd(n)` computes the rolling sample standard deviation over the last
`n` samples.  It shares the same `start_policy` default (`"strict"`) and
callable interface as `RollingMean`.

In [ ]:
std20 = RollingStd(20)(price)

# Warm-up: first 19 outputs are NaN (window not yet full)
assert np.all(np.isnan(std20[:19])), "strict warm-up: first 19 values are NaN"
assert not np.isnan(std20[19]),      "first finite std at index 19"

# Cross-check against pandas
pd_std = pd.Series(price).rolling(20).std().to_numpy()
np.testing.assert_allclose(std20, pd_std, equal_nan=True)

print(f"RollingStd(20) range: {np.nanmin(std20):.3f} - {np.nanmax(std20):.3f}")
print("screamer RollingStd == pandas rolling(20).std() ✓")

## EW family: mean, std, and z-score

The EW family mirrors the rolling family.  `EwStd` computes the
exponentially-weighted standard deviation with the same `span` convention.

In [ ]:
ew_mean = EwMean(span=30)(price)
ew_std  = EwStd(span=30)(price)

# coefficient of variation (relative volatility)
cv = ew_std / ew_mean
print("EwStd / EwMean range:", np.nanmin(cv).round(4), "-", np.nanmax(cv).round(4))

**Takeaways**

- screamer's rolling / EW family is broad: mean, std, zscore, variance,
  skew, kurtosis, min/max, and more - all sharing the same callable interface.
- Window size and start policy are constructor arguments; calling the object
  is always `functor(data)` regardless of the estimator.
- `"strict"` (default) matches pandas `rolling().mean()` exactly.
- EW estimators weight recent data more and respond faster to regime changes;
  rolling estimators treat all samples in the window equally.